# Planning with the final grid-start adapter

Compare raw I-JEPA MSE with the final grid-start adapter on the same 20 held-out cases. The two conditions share start and goal coordinates, seeds, the world model, and the 100-simulator-step budget. The cases cover both rooms, with five same-room cases in each room and five cases in each direction through the doorway.

The notebook verifies the Band 8, Round 3 adapter against its saved checkpoint metadata. It writes 20 MP4s per condition to Drive and reports paired outcomes and crossing counts. Choose a GPU runtime and select **Run all**.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
SOURCE_RUN_ID = '20261109T212006767491Z'
RESUME_EVAL_RUN_ID = ''
RUN_ID = RESUME_EVAL_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE_RUN = DRIVE/'datasets/curriculum_cost_adapter/experience_mpc_route_feedback_full_grid_starts_v1'/SOURCE_RUN_ID
OUTPUT = DRIVE/'datasets/curriculum_cost_adapter/grid_route_planning_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-grid-route-planning')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='9d0a80afcd5ea73e2460f48a3f62280c4249adec'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-square-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
print('Source run:', SOURCE_RUN)
print('Evaluation run ID:', RUN_ID)
print('Results and videos on Drive:', OUTPUT)

In [ ]:
print('GPU:', torch.cuda.get_device_name())
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

In [ ]:
shell([sys.executable, '-m', 'src.mylonas_ijepa_wm.grid_route_planning',
       '--source-run', SOURCE_RUN, '--output-dir', OUTPUT,
       '--encoder-checkpoint', ENCODER, '--world-model-checkpoint', CHECKPOINT,
       '--dino-wm-root', DINO_ROOT, '--device', 'cuda'], cwd=REPO)

In [ ]:
import pandas as pd
from IPython.display import Video, display

rows = pd.DataFrame(json.loads((OUTPUT/'comparison.json').read_text()))
rows.to_csv(OUTPUT/'comparison.csv', index=False)
for label, group in [('Overall', rows), *rows.groupby('route', sort=False)]:
    print(f'{label} ({len(group)} paired cases)')
    display(pd.DataFrame({
        'successes': [int(group[f'{c}_success'].sum()) for c in ('raw', 'adapted')],
        'crossings': [int(group[f'{c}_crossed'].sum()) for c in ('raw', 'adapted')],
        'mean steps': [group[f'{c}_micro_steps'].mean() for c in ('raw', 'adapted')],
        'mean final distance': [group[f'{c}_final_goal_distance'].mean() for c in ('raw', 'adapted')],
    }, index=['raw', 'adapted']).round(2))
adapted_only = rows.adapted_success & ~rows.raw_success
raw_only = rows.raw_success & ~rows.adapted_success
print('Adapted-only successes:', int(adapted_only.sum()), '| Raw-only successes:', int(raw_only.sum()))
display(rows[['case_id', 'route', 'start', 'goal', 'raw_success', 'adapted_success',
              'raw_micro_steps', 'adapted_micro_steps']])
print('Raw videos:', OUTPUT/'raw'/'videos')
print('Adapted videos:', OUTPUT/'adapted'/'videos')

example = rows[adapted_only].iloc[0] if adapted_only.any() else rows[rows.route.eq('left_to_right')].iloc[0]
print('Preview case:', example.case_id, example.route)
for condition in ('raw', 'adapted'):
    print(condition)
    display(Video(example[f'{condition}_video_path'], embed=True, width=224))